# Day 4 — Python → C++ Code Generator

Use a frontier model to convert Python to high-performance C++.

## Quick start
1. Run the **Prerequisite Check** cell first.
2. Fill in your `.env` (see check cell for the format).
3. Run all cells top-to-bottom.
4. The Gradio UI launches in your browser automatically.


In [1]:
# ── PREREQUISITE CHECK ──────────────────────────────────────────────────────
# Run this cell first.  It verifies every dependency and prints install hints.

import shutil, sys, subprocess, importlib
from pathlib import Path

ok = True

# 1. Python packages
for pkg, cmd in {
    "openai":  "pip install openai",
    "dotenv":  "pip install python-dotenv",
    "gradio":  "pip install gradio",
    "IPython": "pip install ipython",
}.items():
    try:
        importlib.import_module(pkg)
        print(f"✅ Python package '{pkg}' is installed")
    except ImportError:
        print(f"❌ Missing: '{pkg}'  →  run:  {cmd}")
        ok = False

# 2. C++ compiler
compiler_found = False
for compiler in ("clang++", "g++"):
    if shutil.which(compiler):
        ver = subprocess.run(
            [compiler, "--version"], capture_output=True, text=True
        ).stdout.splitlines()[0]
        print(f"✅ C++ compiler: {compiler}  ({ver})")
        compiler_found = True
        break
if not compiler_found:
    print("❌ No C++ compiler found.")
    print("   Windows options (pick one):")
    print("   • LLVM/Clang  – winget install LLVM.LLVM")
    print("   • MSYS2/GCC   – winget install MSYS2.MSYS2")
    print("     then inside MSYS2:  pacman -S mingw-w64-ucrt-x86_64-gcc")
    ok = False


✅ Python package 'openai' is installed
✅ Python package 'dotenv' is installed
✅ Python package 'gradio' is installed
✅ Python package 'IPython' is installed
✅ C++ compiler: clang++  ((built by Brecht Sanders, r2) clang version 19.1.1)


In [2]:
# ── IMPORTS ─────────────────────────────────────────────────────────────────
# sys.path ensures 'src' is importable when the notebook lives at project root.
import sys, os, platform
sys.path.insert(0, ".")   # project root → enables  from src.xxx import ...

import gradio as gr
from IPython.display import Markdown, display

# Shared utilities extracted from all three notebooks
from src.code_utils  import run_python, write_output, strip_fences
from src.compiler    import pick_cpp_compiler, get_cpp_commands, compile_and_run_cpp
from src.llm_client  import build_clients, call_llm, check_required_keys
from system_info     import retrieve_system_info
from styles          import CSS

print("✅ All imports OK")


✅ All imports OK


In [3]:
# ── API KEY LOADING ──────────────────────────────────────────────────────────
keys = check_required_keys(["GROQ_API_KEY", "OPENROUTER_API_KEY"])
print(f"✅ GROQ_API_KEY     : ...{keys['GROQ_API_KEY'][-6:]}")
print(f"✅ OPENROUTER_API_KEY: ...{keys['OPENROUTER_API_KEY'][-6:]}")


✅ GROQ_API_KEY     : ...Md5wA9
✅ OPENROUTER_API_KEY: ...464067


In [5]:
# ── LLM CLIENTS (OpenAI-compatible wrappers) ─────────────────────────────────
# All providers are accessed through the same openai.OpenAI SDK.
# NOTE: reasoning_effort is OpenAI-only and must NOT be passed to Groq/OpenRouter.

_clients = build_clients()
groq       = _clients["groq"]
openrouter = _clients["openrouter"]

# Model → client routing
# qwen3 and qwen3-coder live on OpenRouter; llama on Groq.
models = [
    "llama-3.3-70b-versatile",
    "qwen/qwen3-32b",
    "qwen/qwen3-coder-30b-a3b-instruct",
]
clients = {
    "llama-3.3-70b-versatile":           groq,
    "qwen/qwen3-32b":                    openrouter,
    "qwen/qwen3-coder-30b-a3b-instruct": openrouter,
}

print("✅ Clients initialized")
print("   Models available:")
for m in models:
    provider = "Groq" if clients[m] is groq else "OpenRouter"
    print(f"     {m}  [{provider}]")


✅ Clients initialized
   Models available:
     llama-3.3-70b-versatile  [Groq]
     qwen/qwen3-32b  [OpenRouter]
     qwen/qwen3-coder-30b-a3b-instruct  [OpenRouter]


In [6]:
# ── SYSTEM INFO ──────────────────────────────────────────────────────────────
# Passed to the LLM so it can emit compiler flags tuned to this machine.
system_info = retrieve_system_info()
print("✅ System info collected")
system_info


✅ System info collected


{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '11',
  'version': '10.0.26200',
  'kernel': '11',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-w64-windows-gnu'},
 'package_managers': ['winget'],
 'cpu': {'brand': '12th Gen Intel(R) Core(TM) i5-1235U',
  'cores_logical': 12,
  'cores_physical': 10,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'gcc.EXE (MinGW-W64 x86_64-ucrt-posix-seh, built by Brecht Sanders, r2) 14.2.0',
   'g++': 'g++.EXE (MinGW-W64 x86_64-ucrt-posix-seh, built by Brecht Sanders, r2) 14.2.0',
   'clang': '(built by Brecht Sanders, r2) clang version 19.1.1',
   'msvc_cl': ''},
  'build_tools': {'cmake': 'cmake version 3.30.4',
   'ninja': '1.12.1',
   'make': ''},
  'linkers': {'ld_lld': '(built by Brecht Sanders, r2) LLD 19.1.1 (compatible with GNU linkers)'}}}

In [7]:
# ── COMPILER SETUP ───────────────────────────────────────────────────────────
compiler = pick_cpp_compiler()
compile_command, run_command = get_cpp_commands(compiler)

print(f"Compiler : {compiler}")
print(f"Compile  : {' '.join(compile_command)}")
print(f"Run      : {' '.join(run_command)}")


Compiler : clang++
Compile  : clang++ -O3 -march=native -o main.exe main.cpp
Run      : main.exe


In [8]:
# ── PROMPT TEMPLATES ─────────────────────────────────────────────────────────
SYSTEM_PROMPT = """
Your task is to convert Python code into high-performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ code must produce identical output in the shortest possible time.
Do NOT wrap the output in markdown fences.
"""

def user_prompt_for(python: str) -> str:
    return f"""Port this Python code to C++ with the fastest possible implementation.
System information:
{system_info}

Your response will be written to main.cpp and compiled with:
{compile_command}

Respond only with C++ code.  Do NOT wrap in markdown fences.

Python code to port:

```python
{python}
```"""

def messages_for(python: str) -> list:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt_for(python)},
    ]


In [9]:
# ── CORE PORT FUNCTION ───────────────────────────────────────────────────────
def port(model: str, python: str) -> str:
    """Convert Python → C++ via the selected model, write to main.cpp."""
    client = clients[model]
    cpp = call_llm(client, model, messages_for(python))
    write_output(cpp, "main.cpp")
    return cpp

def compile_and_run() -> str:
    """Compile main.cpp (already on disk) and run the binary."""
    return compile_and_run_cpp(compile_command, run_command)


In [10]:
# ── SAMPLE PYTHON PROGRAM (Pi approximation) ─────────────────────────────────
pi_code = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * param1 - param2
        result -= (1 / j)
        j = i * param1 + param2
        result += (1 / j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""


In [11]:
# ── GRADIO UI ────────────────────────────────────────────────────────────────
def convert_and_run(model: str, python_code: str):
    """Generate C++, compile, run.  Returns (cpp_source, run_output)."""
    cpp = port(model, python_code)
    output = compile_and_run()
    return cpp, output


with gr.Blocks(theme=gr.themes.Soft(), css=CSS) as ui:
    gr.Markdown("# 🚀 Python → C++ Code Converter")

    # Code panels
    with gr.Row():
        with gr.Column(scale=1):
            python_input = gr.Code(
                label="Source: Python",
                language="python",
                lines=25,
                value=pi_code,
                elem_classes=["card"],
            )
        with gr.Column(scale=1):
            cpp_output = gr.Code(
                label="Output: C++",
                language="cpp",
                lines=25,
                elem_classes=["card"],
            )

    # Controls
    with gr.Row(variant="panel"):
        with gr.Column(scale=4):
            model_select = gr.Dropdown(
                choices=models,
                label="Intelligence Engine",
                value=models[0],
            )
        with gr.Column(scale=1, min_width=140):
            convert_btn = gr.Button(
                "🔄 Convert", variant="primary", size="lg",
                elem_classes=["convert-btn"],
            )
        with gr.Column(scale=1, min_width=140):
            run_btn = gr.Button(
                "▶ Run C++", variant="secondary", size="lg",
                elem_classes=["run-btn", "cpp"],
            )

    # Output panels
    with gr.Row():
        with gr.Column(scale=1):
            python_out = gr.Textbox(
                label="Python Output",
                lines=4,
                interactive=False,
                elem_classes=["py-out"],
            )
        with gr.Column(scale=1):
            cpp_run_out = gr.Textbox(
                label="C++ Output",
                lines=4,
                interactive=False,
                elem_classes=["cpp-out"],
            )

    # Wiring
    convert_btn.click(
        fn=convert_and_run,
        inputs=[model_select, python_input],
        outputs=[cpp_output, cpp_run_out],
    )
    convert_btn.click(
        fn=run_python,
        inputs=[python_input],
        outputs=[python_out],
    )
    run_btn.click(
        fn=compile_and_run,
        inputs=[],
        outputs=[cpp_run_out],
    )

ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
